# 3. 编码注意机制

In [1]:
# 从标准库的 importlib.metadata 模块导入 version 函数
# 这个函数可以用来查询已安装 Python 包的版本号
from importlib.metadata import version

# 打印 PyTorch 框架的版本号
# version("torch") 会返回当前环境中安装的 torch 版本字符串
print("torch version:", version("torch"))

torch version: 2.3.1


## 3.3.1

> Step 1：计算未归一化的注意力分数

In [2]:
import torch

# 输入是一个 6×3 的张量
# 每一行代表一个词的 3 维嵌入向量
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your   (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with   (x^4)
        [0.77, 0.25, 0.10],  # one    (x^5)
        [0.05, 0.80, 0.55]   # step   (x^6)
    ]
)

In [4]:
# 选择第2个输入词元（"journey"）作为查询
query = inputs[1]  # 对应 x^(2)

# 初始化一个空张量来存储注意力分数
attn_scores_2 = torch.empty(inputs.shape[0])
print(inputs.shape)
attn_scores_2

torch.Size([6, 3])


tensor([0., 0., 0., 0., 0., 0.])

In [6]:
# 遍历所有输入词元，计算点积得到未归一化注意力分数
for i, x_i in enumerate(inputs):
    # 计算查询与每个输入向量的点积
    attn_scores_2[i] = torch.dot(x_i, query)
print(0.43 * 0.55 + 0.15 * 0.87 + 0.89 * 0.66)
attn_scores_2

0.9544


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

点积本质上是两个向量按元素相乘后求和的简写形式。

In [7]:
res = 0.
# 手动计算点积：按元素相乘后累加
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
# 使用 PyTorch 内置函数计算点积
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


>步骤 2：归一化注意力分数

对未归一化的注意力分数（ω）进行归一化，使它们的总和为 1。

In [8]:
# 简单归一化：分数除以所有分数的总和
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


 所有权重的和为 1，且每个权重代表对应输入词元的相对重要性。

然而在实践中，更常用且推荐的是 Softmax 函数进行归一化，它能更好地处理极端值，并且在训练时拥有更理想的梯度特性。

In [9]:
def softmax_naive(x):
    # Softmax 公式：对每个元素取指数后，除以所有元素指数的和
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


 因此在实际应用中，建议使用 PyTorch 内置的 softmax 函数，它经过了高度优化，能避免数值问题。

In [10]:
# 使用 PyTorch 内置的 Softmax 函数对注意力分数进行归一化
# dim=0 表示在第0个维度（即输入序列维度）上进行归一化
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


>步骤 3：将每个输入词元的嵌入向量 x(i) 与对应的注意力权重相乘，再将所有结果向量相加，得到上下文向量 z(2)。

In [11]:
# 选择第2个输入词元作为查询
query = inputs[1]  # 对应 "journey"

# 初始化上下文向量，维度与查询向量一致
context_vec_2 = torch.zeros(query.shape)

# 遍历所有输入词元，计算加权和
for i, x_i in enumerate(inputs):
    # 每个输入向量乘以对应的注意力权重，再累加到上下文向量
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


这个 3 维向量就是以 “journey” 为查询时得到的上下文向量，它融合了所有输入词元的信息。

# 总结

1. 注意力机制核心
- 动态为输入词元分配权重，让模型选择性关注关键信息
- 自注意力让序列中每个位置能与所有位置交互，捕捉全局依赖

2. 简化自注意力计算流程
- 点积计算未归一化注意力分数（衡量相似度）
- Softmax 归一化得到权重（总和为 1）
- 加权求和得到上下文向量（融合全局信息）

3. 关键技术点
- 点积是相似度计算的基础
- Softmax 保证权重可解释性与训练稳定性
- 上下文向量是输入向量的增强表示